# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

In [38]:
#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import import_datasets
import general_utils
from generate_answer_cot import generate_answer_cot
#Import other answer generation functions as needed
from evaluate_model import evaluate_model
from rag import rag_pipeline
from generate_answer_rag import generate_answer_rag
from generate_answer_std import generate_answer_std

client = general_utils.load_gemini_client()



In [51]:
import json
from rag import build_retriver


def generate_answer_rag(question, choices, client, vector_store, k=1):
    """
    Generates an answer using RAG.
    Args:
        question (str): The question to answer.
        choices (list): A list of answer choices.
        client (Client): The Gemini client.
        vector_store (InMemoryVectorStore): The vector store.
        k (int): The number of most similar documents to return.
    Returns:
        str: The selected answer choice letter.
        str: The full response from the model for debugging.
    """
    # build the retriever 
    retrived_docs = build_retriver(vector_store, [question], k = k)
    # batch() returns a list of lists, so we need to get the first element
    retrived_docs = retrived_docs[0] if retrived_docs else []
    content_text = [doc.page_content for doc in retrived_docs]
    content = "\n\n".join(content_text)
    
    # Format choices clearly for the model
    formatted_choices = "\n".join([f"{chr(65+i)}) {choice}" for i, choice in enumerate(choices)])
    
    prompt = (
        f"Context: {content}\n\n"
        f"Question: {question}\n\n"
        f"Choices:\n{formatted_choices}\n\n"
        "Provide your final answer as one of the letters A, B, C, or D."
    )

    # Use JSON mode to guarantee a parseable structure
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": {
                "type": "object",
                "properties": {
                    "reasoning": {"type": "string"},
                    "answer": {"type": "string", "enum": ["A", "B", "C", "D"]}
                },
                "required": ["reasoning", "answer"]
            },
            "max_output_tokens": 5096,
            "temperature": 0.1, # Low temperature for factual MMLU tasks
        }
    )
    data = json.loads(response.text)
    return data["answer"], data["reasoning"]
    
import pandas as pd

def evaluate_model(df, answer_generation_function, client, debugging=False, vector_store=None, k=1):
    """
    Evaluates the model on a given DataFrame of questions.
    Args:
        df (pd.DataFrame): DataFrame containing 'question', 'choices', and 'answer' columns.
        answer_generation_function (function): Function to generate answers given a question and choices.
                                        The function should take in (question: str, choices: List[str], client) 
                                        and return a tuple (answer_choice: str, explanation: str).
        debugging (bool): If True, prints detailed information for each question.
    Returns:
        float: Accuracy of the model.
        pd.DataFrame: DataFrame containing results for each question.
    """
    correct_answers = 0
    total_questions = len(df)
    results_df = pd.DataFrame(columns=['question', 'model_answer', 'correct_answer'])
    for index, row in df.iterrows():
        question = row['question']
        choices = [row['choices'][0], row['choices'][1], row['choices'][2], row['choices'][3]]
        correct_choice = row['answer']
        converted_choices = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
        correct_choice = converted_choices[correct_choice]
        correct = False
        subject = row['subject']
        
        # Check if function name contains "rag" to pass additional parameters
        if "rag" in answer_generation_function.__name__.lower():
            kwargs = {"client": client, "vector_store": vector_store, "k": k}
        else:
            kwargs = {"client": client}
        
        try:
            model_answer = answer_generation_function(question, choices, **kwargs)

        except:
            #try again in case of an error
            try:
                model_answer = answer_generation_function(question, choices, **kwargs)
            except:
                model_answer = ("Error", "Error")

        if model_answer[0] == correct_choice:
            correct_answers += 1
            correct = True

        results_df = pd.concat([results_df, pd.DataFrame({
            'question': [question],
            'model_answer': [model_answer[0]],
            'model_reasoning': [model_answer[1]],
            'correct_answer': [correct_choice],
            'correct': [correct],
            'subject': [subject]
        })], ignore_index=True)
        if debugging:
            print(f"Q{index}: {question}")
            print(f"Model Answer: {model_answer[0]}, Correct Answer: {correct_choice}\n")

    accuracy = correct_answers / total_questions
    return accuracy, results_df

## RAG Implementation

> Warning: takes 30 minutes to run!

In [ ]:
# build the rag pipeline to vectorize concept net
file_path = "../input_data/concept_net.txt"
vector_store, ids = rag_pipeline(file_path, chunk_size=2000, chunk_overlap=200, add_start_index=True, batch_processing_size=700)

In [ ]:
#Generate 100 samples from the mmlu datasets
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']
datasets = import_datasets.import_datasets(dataset_names)
df_anatomy = datasets['anatomy']
df_formal_logic = datasets['formal_logic']
df_moral_scenarios = datasets['moral_scenarios']

Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 3360.28 examples/s]


In [ ]:
#Evaluate the accuracy of each model on each dataset. Store the results in dataframes.
#Evaluate the standard model performance on each dataset
accuracy_anatomy_std, results_df_anatomy_std = evaluate_model(df_anatomy, generate_answer_std, client=client)
accuracy_formal_logic_std, results_df_formal_logic_std = evaluate_model(df_formal_logic, generate_answer_std, client=client)
accuracy_moral_scenarios_std, results_df_moral_scenarios_std = evaluate_model(df_moral_scenarios, generate_answer_std, client=client)


In [ ]:
#Evaluate the cot model performance on each dataset
accuracy_anatomy_cot, results_df_anatomy_cot = evaluate_model(df_anatomy, generate_answer_cot, client=client)
accuracy_formal_logic_cot, results_df_formal_logic_cot = evaluate_model(df_formal_logic, generate_answer_cot, client=client)
accuracy_moral_scenarios_cot, results_df_moral_scenarios_cot = evaluate_model(df_moral_scenarios, generate_answer_cot, client=client)

In [ ]:
# WARNING: take ~ 15 minutes to run: Evaluate the RAG model performance on each dataset
accuracy_anatomy_rag, results_df_anatomy_rag = evaluate_model(df_anatomy, generate_answer_rag, client=client, vector_store=vector_store, k=1)
accuracy_formal_logic_rag, results_df_formal_logic_rag = evaluate_model(df_formal_logic, generate_answer_rag, client=client, vector_store=vector_store, k=1)
accuracy_moral_scenarios_rag, results_df_moral_scenarios_rag = evaluate_model(df_moral_scenarios, generate_answer_rag, client=client, vector_store=vector_store, k=1)

In [ ]:
#Combine results for each model into dataframes:
results_df_std = pd.concat([results_df_anatomy_std, results_df_formal_logic_std, results_df_moral_scenarios_std], ignore_index=True)
results_df_cot = pd.concat([results_df_anatomy_cot, results_df_formal_logic_cot, results_df_moral_scenarios_cot], ignore_index=True)
results_df_rag = pd.concat([results_df_anatomy_rag, results_df_formal_logic_rag, results_df_moral_scenarios_rag], ignore_index=True)

In [ ]:
#create a dataframe to summarize the results
results_summary = pd.DataFrame({
    'Dataset': ['Standard', 'CoT', 'RAG'],
    'Anatomy': [accuracy_anatomy_std, accuracy_anatomy_cot, accuracy_anatomy_rag],
    'Formal Logic': [accuracy_formal_logic_std, accuracy_formal_logic_cot, accuracy_formal_logic_rag],
    'Moral Scenarios': [accuracy_moral_scenarios_std, accuracy_moral_scenarios_cot, accuracy_moral_scenarios_rag]
}) 
print(results_summary)

results_summary.to_csv('model_performance_summary.csv', index=False)

    Dataset  Anatomy  Formal Logic  Moral Scenarios
0  Standard     0.35          0.13             0.20
1       CoT     0.00          0.00             0.00
2       RAG     0.17          0.26             0.22


In [ ]:
#Generate a dataframe with 20 randomly selected errors for each model:
def get_error_samples(results_df, model_name):
    errors_df = results_df[results_df['correct'] == False]
    sample_errors_df = errors_df.sample(n=min(20, len(errors_df)), random_state=42)
    sample_errors_df['model'] = model_name
    return sample_errors_df

#Generate a csv file containing at most 20 randomly selected errors for each model
error_samples_std = get_error_samples(results_df_std, 'Standard')
error_samples_cot = get_error_samples(results_df_cot, 'CoT')
error_samples_rag = get_error_samples(results_df_rag, 'RAG')
error_samples_df = pd.concat([error_samples_std, error_samples_cot, error_samples_rag], ignore_index=True)
error_samples_df.to_csv('model_error_samples.csv', index=False)